# 17-620 Lab 2 — Completed Example: Classical and Quantum Similarity

**Reference notebook | PennyLane | Fall 2026 | Ungraded — no submission required**

**This is the completed, runnable version of Lab 2, shared after class for review.** The original `lab.ipynb` remains available if you want to practice filling in the exercises yourself. This notebook demonstrates one correct implementation of each exercise; it is not a homework answer key, and no part of Lab 2 is collected or graded.

### The central question

**When we change how we represent data, how do we change the way we compare it?**

We already know how to compare ordinary vectors. In this lab, we will carry that idea into quantum computing. First, we will use cosine similarity to compare classical vectors. Then we will encode data in quantum states and measure how closely those states match. Finally, we will ask what a machine-learning algorithm could learn from those measurements.

By the end, you should be able to:

1. Explain cosine similarity and squared overlap.
2. Encode classical data using rotation angles or quantum-state amplitudes.
3. Build and compare a quantum kernel matrix for each encoding.
4. Estimate a kernel value from a finite number of measurements.
5. Distinguish a fixed quantum kernel from the coefficients an SVM learns.

**Reading:** *A Practical Guide to Quantum Machine Learning and Quantum Optimization*, Chapter 9.

**How to use this completed example:** Run the cells in order and inspect the implemented encoding, overlap, and sampling functions. You can compare the results with your own work in `lab.ipynb`. The checks demonstrate expected behavior. The discussion questions are prompts to think about or talk through, not written responses you must submit. Work through as much as you can during class and revisit anything useful afterward.

**Nothing from Lab 2 needs to be turned in or uploaded to Canvas.** Your graded work is assessed separately through quizzes and homework.

> **Keep this distinction in mind:** A quantum circuit can compare data without learning from it. We must train a model separately if we want it to make predictions.

In [ ]:
import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

print("PennyLane", qml.__version__)

## Part 1 — How do we compare classical vectors? (10 minutes)

Picture two arrows. They might point in the same direction, meet at a right angle, or point in opposite directions. **Cosine similarity** describes that relationship without depending on the arrows' lengths:

$$
\operatorname{cosSim}(a,b)=\frac{a\cdot b}{\|a\|\|b\|}.
$$

For two nonzero vectors, a similarity of **+1** means the arrows point together; **0** means they are perpendicular; and **−1** means they point in opposite directions.

Run the next cell to compare four vectors. Each square in the heatmap compares **two data points**. Find the square for points 0 and 3. What does its value tell you?

In [ ]:
X = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [-1.0, 0.0]])
cosine = cosine_similarity(X)
print("Classical cosine similarity:\n", np.round(cosine, 3))

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cosine, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(X)), labels=[f"x{i}" for i in range(len(X))])
ax.set_yticks(range(len(X)), labels=[f"x{i}" for i in range(len(X))])
ax.set_title("Classical cosine similarity")
fig.colorbar(im, ax=ax, label="cosine similarity")
plt.show()

### Explore the vectors yourself

Use the four sliders to move the arrows. As you change their components, watch their lengths, angle, dot product, and cosine similarity.

The dot product comes directly from the components:

$$
a\cdot b=a_xb_x+a_yb_y.
$$

Try three arrangements: arrows pointing together, perpendicular arrows, and arrows pointing in opposite directions. Then make one arrow's length zero. **Cosine similarity is undefined for a zero vector**, because the formula divides by its length.

**Question 1.** Why can cosine similarity be negative? Which pairs in the heatmap are perpendicular?

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def classical_cosine_explorer(a_x=4.0, a_y=1.0, b_x=-1.0, b_y=3.0):
    a = np.array([a_x, a_y], dtype=float)
    b = np.array([b_x, b_y], dtype=float)
    dot = float(np.dot(a, b))
    norm_a, norm_b = np.linalg.norm(a), np.linalg.norm(b)
    valid = norm_a > 1e-12 and norm_b > 1e-12
    cosine = float(np.clip(dot / (norm_a * norm_b), -1, 1)) if valid else None
    theta = float(np.degrees(np.arccos(cosine))) if valid else None

    fig, ax = plt.subplots(figsize=(5.8, 4.6))
    for v, label, color in [(a, "a", "tab:blue"), (b, "b", "tab:green")]:
        ax.annotate("", xy=tuple(v), xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", color=color, lw=2.8, mutation_scale=18))
        if np.linalg.norm(v) > 1e-12:
            ax.text(v[0] * 1.08 + 0.1, v[1] * 1.08 + 0.1, label,
                    fontsize=14, color=color, weight="bold")
    ax.axhline(0, color="gray", lw=0.9)
    ax.axvline(0, color="gray", lw=0.9)
    ax.set(xlim=(-7, 7), ylim=(-7, 7), aspect="equal",
           xlabel="x component", ylabel="y component",
           title="Two classical feature vectors")
    ax.grid(alpha=0.22)
    plt.show()

    print(f"Dot product: a·b = ({a_x:g})({b_x:g}) + ({a_y:g})({b_y:g}) = {dot:.3f}")
    print(f"Vector magnitudes: ||a|| = {norm_a:.3f}, ||b|| = {norm_b:.3f}")
    if not valid:
        print("Cosine similarity: undefined (at least one vector is zero).")
        return
    if np.isclose(cosine, 1.0, atol=1e-10):
        direction = "same direction"
    elif np.isclose(cosine, -1.0, atol=1e-10):
        direction = "opposite directions"
    elif np.isclose(cosine, 0.0, atol=1e-10):
        direction = "orthogonal"
    elif cosine > 0:
        direction = "acute angle / positive similarity"
    else:
        direction = "obtuse angle / negative similarity"
    print(f"cosSim(a,b) = {cosine:+.3f} | Angle = {theta:.1f}° | {direction}")

component_controls = {
    "a_x": widgets.FloatSlider(value=4, min=-6, max=6, step=0.5, description="aₓ",
                               continuous_update=False),
    "a_y": widgets.FloatSlider(value=1, min=-6, max=6, step=0.5, description="aᵧ",
                               continuous_update=False),
    "b_x": widgets.FloatSlider(value=-1, min=-6, max=6, step=0.5, description="bₓ",
                               continuous_update=False),
    "b_y": widgets.FloatSlider(value=3, min=-6, max=6, step=0.5, description="bᵧ",
                               continuous_update=False),
}
component_output = widgets.interactive_output(classical_cosine_explorer, component_controls)
display(widgets.VBox([widgets.HTML("<b>Drag to change vector components</b>"),
                      *component_controls.values(), component_output]))

## Part 2 — Why do we square the overlap? (10 minutes)

We can also represent a quantum state as a vector. Unlike the vectors in Part 1, its components may be **complex numbers**.

To compare two quantum states, we take their inner product:

$$
s(x,y)=\langle\phi(y)|\phi(x)\rangle.
$$

We call this result an **overlap amplitude**. It tells us how the states relate mathematically, but it is **not yet a probability**: it may be negative or complex.

The **Born rule** tells us how to calculate the probability of projecting one state onto the other. We take the amplitude's **squared magnitude**:

$$
\boxed{K_Q(x,y)=|\langle\phi(y)|\phi(x)\rangle|^2.}
$$

That operation gives *squared overlap* its name. We square the magnitude of the **inner product**, not the entries of either vector. For example, the amplitude \(0.6+0.3i\) has squared magnitude \(0.6^2+0.3^2=0.45\). For pure states, physicists also call squared overlap **fidelity**.

### How can a circuit measure it?

Suppose the circuit \(U(x)\) prepares the state for data point \(x\). We prepare that state, then apply the **inverse** of the circuit that would prepare the state for \(y\).

The amplitude of returning to the all-zero state is

$$
\langle0\ldots0|U(y)^\dagger U(x)|0\ldots0\rangle=\langle\phi(y)|\phi(x)\rangle.
$$

When we measure, the **probability of getting all zeros** is the squared magnitude of that amplitude. In Part 3, you will build this circuit.

### How does this connect to cosine similarity?

For **nonzero real vectors encoded directly as normalized quantum-state amplitudes**, the inner product of the quantum states equals the classical cosine similarity. Their squared overlap therefore equals

$$
K_Q(x,y)=\operatorname{cosSim}(x,y)^2.
$$

The square removes the sign. For example, opposite real vectors have cosine similarity −1 but squared overlap 1. Their quantum states differ only by a **global phase**, which a measurement cannot distinguish.

**Be careful:** This equality holds for the amplitude encoding just described. **Other encodings produce different kernels.**

**Question 2.** If the overlap amplitude is \(-0.8\), what probability would the circuit measure? Why?

### See what changes when we square a similarity

Keep the teal vector fixed and rotate the orange vector. The display shows two values: their **signed cosine similarity** and its **square**.

At 180°, the vectors point in opposite directions. Their cosine similarity is −1, but their squared overlap is 1.

This visualization models **real amplitude encoding**. It does not describe every quantum feature map.

In [ ]:
import ipywidgets as widgets

def explore_similarity(angle=60):
    theta = np.deg2rad(angle)
    x = np.array([1., 0.])
    y = np.array([np.cos(theta), np.sin(theta)])
    cosine = float(x @ y)
    fidelity = cosine**2
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.arrow(0, 0, x[0], x[1], color="teal", width=0.01, head_width=0.07, length_includes_head=True)
    ax.arrow(0, 0, y[0], y[1], color="darkorange", width=0.01, head_width=0.07, length_includes_head=True)
    ax.set(xlim=(-1.25, 1.25), ylim=(-1.25, 1.25), aspect="equal", xlabel="Feature 1",
           ylabel="Feature 2", title=f"Angle {angle}° | cosine {cosine:+.3f} | squared overlap {fidelity:.3f}")
    ax.grid(alpha=0.25)
    plt.show()

widgets.interact(explore_similarity, angle=widgets.IntSlider(value=60, min=0, max=180, step=5, continuous_update=False));

## Part 3 — Encode data with rotations and compare the resulting states (30 minutes)

We now know what a quantum overlap circuit measures. But first we need to **prepare the states it will compare**.

### Worked example — One input, one rotation

With **angle encoding**, an input value determines a gate's rotation angle. A Y rotation prepares the state

$$
R_Y(\theta)|0\rangle=\cos(\theta/2)|0\rangle+\sin(\theta/2)|1\rangle.
$$

When \(\theta\) is 0, \(\pi/2\), or \(\pi\), the probability of measuring 0 is 1, 1/2, or 0, respectively.

Run the worked example and inspect its circuit diagram. Notice that we use the **input value as a rotation parameter**. We are not necessarily using the geometric angle between two classical vectors.

In [ ]:
worked_dev = qml.device("default.qubit", wires=1)

@qml.qnode(worked_dev)
def worked_rotation(theta):
    qml.RY(theta, wires=0)
    return qml.probs(wires=[0])

for theta in [0.0, np.pi / 2, np.pi]:
    print(f"theta={theta:.3f}: P(0),P(1)={worked_rotation(theta)}")
print(qml.draw(worked_rotation)(np.pi / 2))

### Exercise 3A — Encode two features in two qubits

Each input has two features. Give each qubit a Y rotation controlled by one feature.

The completed cell below uses PennyLane's **AngleEmbedding**, returns probabilities in the order **00, 01, 10, 11**, and runs the checks. Compare this implementation with your own.

**Think:** Why might you rescale real-world features before using them as rotation angles? What happens to the probabilities if you add \(2\pi\) to one angle?

In [ ]:
angle_dev = qml.device("default.qubit", wires=2)

@qml.qnode(angle_dev)
def angle_state(x):
    qml.AngleEmbedding(features=x, wires=[0, 1], rotation="Y")
    return qml.probs(wires=[0, 1])

angle_inputs = np.array([[0.0, 0.0], [np.pi/2, 0.0],
                         [0.0, np.pi/2], [np.pi/2, np.pi/2]])

print("Example:", angle_state(angle_inputs[0]))
print(qml.draw(angle_state)(angle_inputs[1]))
assert np.allclose(angle_state(angle_inputs[0]), [1, 0, 0, 0])
assert np.allclose(angle_state(angle_inputs[3]), [0.25, 0.25, 0.25, 0.25])

### Exercise 3B — Build the angle-encoding quantum kernel

Now that we can prepare an encoded state, we can compare **two** of them.

1. Use **AngleEmbedding** to prepare the state for \(x\).
2. Apply the inverse encoding for \(y\).
3. Measure the probability that both qubits return to **|00⟩**.
4. Repeat the measurement calculation for every pair of inputs to build the **angle kernel matrix**.
5. Plot the matrix and run the checks.

The result is a **squared-overlap probability**. It is not the signed cosine similarity we calculated in Part 1.

**Worked solution:** The next cell implements both encodings and constructs the kernel matrix. Check how the circuit's zero-state probability corresponds to squared overlap.

In [ ]:
kernel_dev = qml.device("default.qubit", wires=2)

@qml.qnode(kernel_dev)
def angle_overlap(x, y):
    qml.AngleEmbedding(features=x, wires=[0, 1], rotation="Y")
    qml.adjoint(qml.AngleEmbedding)(features=y, wires=[0, 1], rotation="Y")
    return qml.expval(qml.Projector([0, 0], wires=[0, 1]))

def make_kernel(points, circuit):
    return np.array([[circuit(x, y) for y in points] for x in points])

K_angle = make_kernel(angle_inputs, angle_overlap)
print(np.round(K_angle, 3))
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(K_angle, vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(4), labels=[f"x{i}" for i in range(4)])
ax.set_yticks(range(4), labels=[f"x{i}" for i in range(4)])
ax.set_title("Quantum angle-overlap kernel")
fig.colorbar(im, ax=ax, label="P(00)")
plt.show()
assert np.allclose(K_angle, K_angle.T, atol=1e-8)
assert np.allclose(np.diag(K_angle), np.ones(4), atol=1e-8)
assert np.all((K_angle >= -1e-8) & (K_angle <= 1 + 1e-8))
assert np.isclose(angle_overlap(angle_inputs[0], angle_inputs[1]), 0.5)

### Why the encoding changes the kernel

We have now used angle encoding to prepare quantum states. Its kernel need not match the one from amplitude encoding.

For **one qubit** with Y-angle encoding, two input angles \(a\) and \(b\) give

$$
K_{\mathrm{angle}}(a,b)=\cos^2((a-b)/2).
$$

By contrast, **real amplitude encoding** gives \(\cos^2\theta\), where \(\theta\) is the **geometric angle between the classical vectors**.

The plot below compares these expressions. Its horizontal axis uses the same numerical angle, but that angle has **different meanings** in the two encodings: a gate-parameter difference in one case and a vector angle in the other.

**Question 3.** What does each kernel return when its angle is 180°? How does the choice of encoding explain the difference?

In [ ]:
delta = np.linspace(0, np.pi, 361)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(np.degrees(delta), np.cos(delta), label="Cosine similarity")
ax.plot(np.degrees(delta), np.cos(delta)**2, label="Amplitude kernel: cos²(delta)")
ax.plot(np.degrees(delta), np.cos(delta/2)**2, label="RY angle kernel: cos²(delta/2)")
ax.set(xlabel="Angle / rotation difference (degrees)", ylabel="Similarity / squared overlap")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

## Part 4 — Build the kernel again with amplitude encoding (15 minutes)

In Part 3, we used each input feature to control a **rotation**. Now we will use the four components of a real vector as the **amplitudes** of a two-qubit state.

We will still measure squared overlap. What changes is **how we prepare each state**.

Follow the same procedure:

1. Use **AmplitudeEmbedding** to prepare the state for vector \(a\).
2. Apply the inverse preparation for vector \(b\).
3. Measure the probability of **|00⟩** to calculate one amplitude-kernel entry.
4. Use the existing helper to build the **entire amplitude kernel matrix**.
5. Plot that matrix beside the **squared classical cosine-similarity matrix**. Keep the color scales identical.

For **nonzero real inputs**, the matrices should match. But this result does not demonstrate a quantum advantage: a classical computer can calculate squared cosine similarity directly, and preparing arbitrary amplitude states can be expensive.

**Exercise 4.** Complete the amplitude circuit and both kernel matrices. Why do a vector and its negative have squared overlap 1?

In [ ]:
amp_dev = qml.device("default.qubit", wires=2)

@qml.qnode(amp_dev)
def amplitude_overlap(a, b):
    qml.AmplitudeEmbedding(a, wires=[0, 1], normalize=True)
    qml.adjoint(qml.AmplitudeEmbedding)(b, wires=[0, 1], normalize=True)
    return qml.expval(qml.Projector([0, 0], wires=[0, 1]))

A = np.array([[1., 0., 0., 0.], [0., 1., 0., 0.],
              [1., 1., 0., 0.], [-1., 0., 0., 0.]])

K_amplitude = make_kernel(A, amplitude_overlap)
K_classical_squared = cosine_similarity(A)**2
print("Quantum amplitude kernel:\n", np.round(K_amplitude, 3))
print("Classical cosine²:\n", np.round(K_classical_squared, 3))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, matrix, title in zip(axes, [K_amplitude, K_classical_squared],
                             ["Quantum amplitude kernel", "Classical cosine²"]):
    im = ax.imshow(matrix, vmin=0, vmax=1, cmap="viridis")
    ax.set_title(title)
    ax.set_xticks(range(4), labels=[f"a{i}" for i in range(4)])
    ax.set_yticks(range(4), labels=[f"a{i}" for i in range(4)])
fig.colorbar(im, ax=axes)
plt.show()

assert np.allclose(K_amplitude, K_classical_squared, atol=1e-7)
assert np.isclose(K_amplitude[0, 3], 1.0)

## Part 5 — Estimate overlap with measurements (10 minutes)

Our simulator has given us exact probabilities. **Hardware gives us measurement outcomes**, one shot at a time.

To estimate a kernel entry, we repeat the overlap circuit and count the times it returns **00**:

$$
\widehat K(x,y)=\frac{\text{number of 00 outcomes}}{\text{number of shots}}.
$$

This fraction estimates the **same kernel value** we calculated earlier. More shots generally reduce the estimate's sampling uncertainty. They do not change the encoding or the kernel's definition.

**Exercise 5.** Complete the circuit, collect 1,000 outcomes, and compare the fraction of 00 outcomes with the exact angle-kernel value.

In [ ]:
shots = 1000
sample_dev = qml.device("default.qubit", wires=2)

@qml.set_shots(shots=shots)
@qml.qnode(sample_dev)
def sampled_angle_overlap(x, y):
    qml.AngleEmbedding(features=x, wires=[0, 1], rotation="Y")
    qml.adjoint(qml.AngleEmbedding)(features=y, wires=[0, 1], rotation="Y")
    return qml.sample(wires=[0, 1])

x, y = angle_inputs[0], angle_inputs[1]
samples = sampled_angle_overlap(x, y)
estimate = np.mean(np.all(samples == 0, axis=1))
print("Estimate:", estimate, "Exact:", angle_overlap(x, y))
assert samples.shape == (shots, 2)
assert abs(estimate - angle_overlap(x, y)) < 0.12

## Part 6 — What would an SVM learn from this kernel? (5 minutes)

**Our quantum circuits have not learned anything.** We chose both encodings before seeing any labels. Each encoding defines how the circuit compares two inputs. The resulting kernel is **fixed**, although its estimate can vary when we use finite shots.

A **support vector machine (SVM)** could use that kernel matrix to learn a decision boundary from labeled examples. A **quantum-kernel SVM (QSVM)** still uses a **classical SVM optimizer**. The quantum circuit supplies similarities; the SVM learns how to use them.

For binary labels \(y_i\in\{-1,+1\}\), a soft-margin SVM chooses coefficients \(\alpha_i\) to maximize

$$
\max_{\alpha}\;\sum_i\alpha_i-\frac12\sum_{i,j}\alpha_i\alpha_j y_i y_j K(x_i,x_j),
$$

subject to \(0\le\alpha_i\le C\) and \(\sum_i\alpha_i y_i=0\).

In this optimization, **the kernel entries stay fixed**. The **SVM coefficients change** to fit the labeled data.

Could we also train the quantum circuit? Yes. We could add adjustable parameters to the feature map and optimize them with a separate objective. A **variational quantum classifier** takes another approach: it trains circuit parameters to make predictions directly. **We do neither in this lab.**

**Question 6.** If we trained an SVM with today's kernel matrix, which values would the SVM change? Which values would stay fixed?

## Optional recap — Check your understanding

Use these questions to reflect on the lab or discuss with a classmate. **You do not need to write or submit answers.**

1. **Classical similarity:** What does cosine similarity measure? Why can its value be negative?
2. **Quantum measurement:** Why do we take the squared magnitude of an overlap amplitude? What outcome does our inverse circuit count?
3. **Encoding:** Why can angle and amplitude encoding produce different kernels? When does amplitude squared overlap equal cosine similarity squared?
4. **Learning:** What remains fixed in this lab? What would an SVM learn? How do finite shots affect our estimates?

**Optional extension:** Give a precomputed kernel matrix to a scikit-learn SVM. Compare its results with a classical kernel on the **same training and test split**. Good accuracy alone does not establish quantum advantage.

**No submission required:** Lab 2 is an ungraded learning activity. You may save your notebook for your own reference, but you do not need to upload it to Canvas. Quizzes and homework have their own separate submission instructions.